# 02. 시계열·센서 데이터 분류

KAMP 제조/설비 센서 데이터처럼 **시간순 센서 값 + 정상/이상(또는 상태) 라벨** 형태를 가정합니다. 핵심은 **시간 누수 방지**(미래 정보 사용 금지, 시간순 분할)입니다.

In [ ]:
import sys; sys.path.append("../common")
from utils import *
import numpy as np, pandas as pd
seed_everything()

In [ ]:
DATA_PATH = "../data/sensor.csv"
TIME_COL  = "timestamp"
TARGET    = "label"
WINDOW    = 30      # 윈도우 크기(행 수)
STEP      = 10      # 윈도우 이동 간격
OUT_PATH  = "../outputs/submission.csv"

In [ ]:
import os
if os.path.exists(DATA_PATH):
    df = read_csv_auto(DATA_PATH, parse_dates=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)
else:
    print("⚠ 데이터 없음 → 합성 데모 센서 데이터 사용")
    n = 6000; t = np.arange(n)
    base = np.sin(t / 50)
    lab = ((t // 400) % 5 == 4).astype(int)       # 400스텝마다 이상 구간
    df = pd.DataFrame({TIME_COL: pd.date_range("2024-01-01", periods=n, freq="min"),
                       "s1": base + np.random.randn(n) * (0.2 + 0.4 * lab),
                       "s2": np.cos(t / 30) + np.random.randn(n) * 0.2 + lab * 0.5,
                       "s3": np.random.randn(n).cumsum() * 0.01, TARGET: lab})
sensors = [c for c in df.select_dtypes("number").columns if c != TARGET]
print(df.shape, sensors); df.head()

## 윈도우 특징 추출 (평균·표준편차·최대/최소·변화량)

In [ ]:
def make_windows(df, window=WINDOW, step=STEP):
    X, y, t = [], [], []
    v = df[sensors].to_numpy()
    for s in range(0, len(df) - window + 1, step):
        w = v[s:s + window]
        f = np.concatenate([w.mean(0), w.std(0), w.max(0), w.min(0), w[-1] - w[0],
                            np.abs(np.diff(w, axis=0)).mean(0)])
        X.append(f); t.append(df[TIME_COL].iloc[s + window - 1])
        y.append(df[TARGET].iloc[s:s + window].mode()[0])   # 윈도우 내 최빈 라벨
    cols = [f"{s}_{k}" for k in ("mean", "std", "max", "min", "delta", "absdiff") for s in sensors]
    return pd.DataFrame(X, columns=cols), np.array(y), np.array(t)
X, y, t = make_windows(df); print(X.shape, np.bincount(y))

## 시간순 분할 검증 (랜덤 분할 금지)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
scores = []
for tr, va in TimeSeriesSplit(5).split(X):
    m = HistGradientBoostingClassifier(random_state=42).fit(X.iloc[tr], y[tr])
    scores.append(f1_score(y[va], m.predict(X.iloc[va]), average="macro"))
print("f1_macro:", np.round(scores, 3), "mean", np.mean(scores).round(3))

In [ ]:
final = HistGradientBoostingClassifier(random_state=42).fit(X, y)
# 테스트 구간이 주어지면 같은 make_windows로 특징을 만든 뒤 final.predict 하고 save_submission으로 저장

## 개선 아이디어
- 윈도우/스텝 튜닝, lag·rolling 특징, FFT 주파수 특징
- 1D-CNN/LSTM은 CPU·시간 제약상 트리 모델 베이스라인 이후에만
- 이상 탐지형이면 IsolationForest/재구성 오차도 비교